In [16]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import Agent, Runner, trace, function_tool, OpenAIChatCompletionsModel, input_guardrail, GuardrailFunctionOutput
from typing import Dict
from pydantic import BaseModel
import os

In [2]:
load_dotenv(override=True)

True

In [ ]:
openai_api_key = os.getenv("OPENAI_API_KEY")
google_api_key = os.getenv("GOOGLE_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
upstage_api_key = os.getenv("UPSTAGE_API_KEY")

if openai_api_key:
    print(f"OpenAI API key exists and starts with {openai_api_key[:8]}")
else:
    print(f"OpenAI API key not set")

if google_api_key:
    print(f"Google API key exists and starts with {google_api_key[:6]}")
else:
    print(f"Google API key not set")

if groq_api_key:
    print(f"Groq API key exists and starts with {groq_api_key[:6]}")
else:
    print(f"Groq API key not set")

if openrouter_api_key:
    print(f"Openrouter API key exists and starts with {openrouter_api_key[:6]}")
else:
    print(f"Openrouter API key not set")

if upstage_api_key:
    print(f"Upstage API key exists and starts with {upstage_api_key[:6]}")
else:
    print(f"Upstage API key not set")

OpenAI API key exists and starts with sk-proj-
Google API key exists and starts with AIzaSy
Groq API key exists and starts with gsk_3E
Openrouter API key exists and starts with sk-or-
Upstage API key exists and starts with up_sHJ


In [4]:
# Gemini (Google)
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

# Groq
GROQ_BASE_URL = "https://api.groq.com/openai/v1"

# Upstage
UPSTAGE_BASE_URL = "https://api.upstage.ai/v1/solar"

# OpenRouter
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

In [11]:
gemini_client = AsyncOpenAI(base_url=GEMINI_BASE_URL, api_key=google_api_key)
groq_client = AsyncOpenAI(base_url=GROQ_BASE_URL, api_key=groq_api_key)
upstage_client = AsyncOpenAI(base_url=UPSTAGE_BASE_URL, api_key=upstage_api_key)
claude_client = AsyncOpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

gemini_model = OpenAIChatCompletionsModel(model="gemini-2.5-flash", openai_client=gemini_client)
groq_model = OpenAIChatCompletionsModel(model="openai/gpt-oss-120b", openai_client=groq_client)
upstage_model = OpenAIChatCompletionsModel(model="solar-pro", openai_client=upstage_client)
claude_model = OpenAIChatCompletionsModel(model="anthropic/claude-sonnet-4-5", openai_client=claude_client)

In [12]:
instructions1 = "You are a professional sales agent for ComplAI that sells SOC2 compliance SaaS tool for compliance and audits,\
                powered by AI. You write serious professional emails"
instructions2 = "You are a humorous engaging sales agent for ComplAI that sells SOC2 compliance SaaS tool for compliance and audits, \
                 powered by AI. You write fun engaging emails"
instructions3 = "You are a busy concise sales agent ComplAI that sells SOC2 compliance SaaS tool for compliance and audits, \
                 powered by AI. You write short punchy emails"


sales_agent1 = Agent(name="Professional Sales Agent", instructions=instructions1, model=gemini_model)
sales_agent2 = Agent(name="Engaging Sales Agent", instructions=instructions2, model=groq_model)
sales_agent3 = Agent(name="Busy Sales Agent", instructions=instructions3, model=upstage_model)

In [14]:
description = "Write a cold sales email"

tool1 = sales_agent1.as_tool(tool_name="sales_agent1", tool_description=description)
tool2 = sales_agent2.as_tool(tool_name="sales_agent2", tool_description=description)
tool3 = sales_agent3.as_tool(tool_name="sales_agent3", tool_description=description)

tools = [tool1, tool2, tool3]

In [18]:
@function_tool
def send_html_email(subject:str, html_body:str)-> Dict[str, str]:
    """Send out email with give subject and html body to all sales prospects"""
    print(f"Subject: {subject}\n{html_body}")
    return{"status": "success"}


In [22]:
subject_instructions = "You can write a cold sales email. You are given a message, write a subject for an email that will likey get a response from the customer."
html_instructions = "You convert a text email body into a HTML email body. You are give an text email body, convert it into clean, simple, compelling \
                     layout and design"

subject_writer = Agent(name="Subject Writer", instructions=subject_instructions, model=claude_model)
subject_tool =subject_writer.as_tool(tool_name="subject_writer", tool_description="Write a subject line for cold sales email")

html_converter = Agent(name="HTML Converter", instructions=html_instructions, model=claude_model)
html_tool =html_body.as_tool(tool_name="html_converter", tool_description="Convert the text email body into HTML body")

In [24]:
email_tools=[send_html_email, subject_tool, html_tool]

In [25]:
email_instructions="You format and send emails. First use the subject_tool to write a subject for a cold sales email. \
                    Then use the html_converter tool to convert the email to HTML body.\
                    Tnen send out the email to all sales prospects"


emailer_agent= Agent(
    name="Email Agent",
    instructions=email_instructions,
    tools=email_tools,
    handoff_description="Convert email to HTML and send",
    model="gpt-4o-mini"
)

In [34]:
# Build Guardrail

class NameCheckOutput(BaseModel):
    is_name_in_message: bool
    name: str

guardrail_agent = Agent(
    name="Guardrail Agent",
    instructions="Check if user includes personal name",
    output_type = NameCheckOutput,
    model="gpt-4o-mini"
)

In [28]:
# input Guardrail

@input_guardrail
async def guardrail_against_name(ctx, agent, message):
    result = await Runner.run(guardrail_agent,message, context=ctx.context)
    is_name_in_message = result.final_output.is_name_in_message
    return GuardrailFunctionOutput(output_info=result.final_output, tripwire_triggered=is_name_in_message)


In [31]:
sales_manager_instructions = "Use all 3 sales agent tools to generate drafts, pick the best one \
                              hand off to Email Agent, Must use tools, not write emails yourself \
                              hand off exactly ONE email"

sales_manager_agent = Agent(
    name="Sales Manager",
    instructions=sales_manager_instructions,
    tools = tools,
    handoffs = [emailer_agent],
    model="gpt-4o-mini",
    input_guardrails=[guardrail_against_name]
)

In [40]:
message = "Send out a cold sales email addressed to Dear CEO from Alice"

with trace("Protected Automated SDR"):
    try:
        result = await Runner.run(sales_manager_agent, message)
        print(result.final_output)
    except Exception as e:
        print(f"BLOCKED: {e}")

BLOCKED: Guardrail InputGuardrail triggered tripwire


In [39]:
message = "Send out a cold sales email addressed to Dear CEO from Head of Business Department"

with trace("Protect Automated SDR"):
    result = await Runner.run(sales_manager_agent, message)
    print(result.final_output)

Subject: Unlock Innovative Solutions for Your Business
<p>Dear CEO,</p>
<p>I hope this message finds you well. I’m reaching out from [Your Company Name] to share some innovative solutions tailored specifically for businesses like yours.</p>
<p>In today’s competitive landscape, companies that leverage advanced strategies can gain a significant edge. Our expertise in [specific services] allows us to support organizations aiming to drive growth and elevate efficiency.</p>
<p>I would love to set up a brief call to explore potential synergies and discuss how we can assist you in achieving your business goals.</p>
<p>Best regards,</p>
<p>[Your Name]<br>Head of Business Department<br>[Your Company Name]<br>[Your Contact Information]</p>
The email has been successfully sent to the CEO with the subject "Unlock Innovative Solutions for Your Business." If there's anything else you need, feel free to ask!
